# NHANES 2003-2004: Predicting 10-Year Cardiovascular Mortality from Accelerometry
## Notebook 01: Feature Engineering

**Author:** David Moreno

**Date:** June 2025

**Objective:** Build clean, analysis-ready dataset from NHANES 2003-2004

**Output:** nhanes_2003_2004_clean.csv





## Notebook overview
This notebook builds a clean, analysis-ready dataset from NHANES 2003-2004 accelerometry and clinical data.  
The target is 10-year cardiovascular mortality.

## Processing strategy
- Python (pandas, numpy) for all data except accelerometry processing  
- R minimal (via rpy2 or separate script) only for rnhanesdata package functions  
- One participant = one row in final dataset  

## Raw data files (manually downloaded to Drive)

| File                                   | Content                                             |
|----------------------------------------|-----------------------------------------------------|
| DEMO_C.xpt                             | Demographics (age, sex, education, income)         |
| BMX_C.xpt                              | Body measurements (weight, height → BMI)           |
| BPX_C.xpt                              | Blood pressure (systolic)                          |
| L13_C.xpt                              | Total cholesterol, HDL cholesterol                 |
| L13AM_C.xpt                            | LDL cholesterol, triglycerides                     |
| L10AM_C.xpt                            | Fasting glucose                                    |
| L10_C.xpt                          | HbA1c (glycohemoglobin) - diabetes marker      |
| NHANES_2003_2004_MORT_2019_PUBLIC.dat  | Mortality + follow-up time                         |
| PAXRAW_C.zip                           | Raw accelerometry (contains paxraw_c.xpt)          |


In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
from google.colab import drive

drive.mount('/content/drive')



In [ ]:
# Project paths
PROJECT_PATH = '/content/drive/MyDrive/NHANES_Accelerometry_Project'
RAW_PATH = f"{PROJECT_PATH}/data/raw"
PROC_PATH = f"{PROJECT_PATH}/data/processed"

# List available raw files
print("Raw files available:")
raw_files = sorted(os.listdir(RAW_PATH))

for f in raw_files:
    print(f"  - {f}")


## 2. Mortality Data Processing

### 2.1 Load Mortality File & Create 10-Year Natural Mortality Target

Based on CDC NHANES Linked Mortality File specifications. Natural mortality excludes accidents (UCOD_LEADING = 4). Ten-year window uses PERMTH_EXM ≤ 120 months, capped at 192 months (maximum possible follow-up for 2003-2004).

In [ ]:
# Install missing package
!pip install pyreadstat

import pandas as pd
import numpy as np
import os
import pyreadstat

RAW_PATH = '/content/drive/MyDrive/NHANES_Accelerometry_Project/data/raw'
PROC_PATH = '/content/drive/MyDrive/NHANES_Accelerometry_Project/data/processed'

mortality_file = os.path.join(RAW_PATH, 'NHANES_2003_2004_MORT_2019_PUBLIC.dat')

# Parse mortality file
records = []
with open(mortality_file, 'r') as f:
    for line in f:
        seqn = int(line[0:5])

        # MORTSTAT: positions 14-15
        mort = line[14:16].strip()
        if mort.startswith('1'):
            mortstat = 1
        elif mort.startswith('2'):
            mortstat = 0
        else:
            mortstat = None

        # UCOD_LEADING: positions 16-19
        ucod = line[16:19].strip()
        if ucod == '..' or ucod == '':
            ucod = None
        elif ucod.isdigit():
            ucod = int(ucod)
        else:
            ucod = None

        # PERMTH_EXM: positions 45-48
        permth = line[45:48].strip()
        if permth.isdigit():
            permth_exm = int(permth)
        else:
            permth_exm = None

        records.append({
            'SEQN': seqn,
            'MORTSTAT': mortstat,
            'UCOD_LEADING': ucod,
            'PERMTH_EXM': permth_exm
        })



In [ ]:
mort = pd.DataFrame(records)
# Filter eligible (known vital status)
eligible = mort[mort['MORTSTAT'].notna()].copy()
eligible['MORTSTAT'] = eligible['MORTSTAT'].astype(int)

# Cap PERMTH_EXM at maximum possible (192 months for 2003-2004)
eligible['PERMTH_EXM'] = eligible['PERMTH_EXM'].clip(upper=192)

# Create target: natural death within 10 years (≤120 months), exclude accidents (code 4)
eligible['target_natural_10yr'] = (
    (eligible['MORTSTAT'] == 1) &
    (eligible['PERMTH_EXM'] <= 120) &
    (eligible['UCOD_LEADING'] != 4)
).astype(int)

print(f"Eligible participants: {len(eligible)}")
print(f"Natural deaths within 10 years: {eligible['target_natural_10yr'].sum()}")
print(f"Event rate: {100 * eligible['target_natural_10yr'].mean():.2f}%")

In [ ]:
eligible.head(20)

## 2.2 Merge with Demographics & Apply Age Filters: VBHC Justification

**Value-Based Healthcare (VBHC) Perspective**

The age filter (20-74 years) is not arbitrary. It responds to a core VBHC principle: measure what matters for preventable, high-value intervention.

- **Premature mortality as a quality metric:** The OECD and WHO define premature mortality as death before age 75. These deaths are more likely to be modifiable through preventive care, lifestyle changes, and early detection—precisely where wearables and blood tests can add value.

- **Avoiding confounding by non-preventable aging:** After age 75, mortality risk increases exponentially regardless of activity levels or biomarkers. Including these deaths would dilute the model's ability to detect signals from modifiable factors, reducing its clinical utility for value-based decision-making.

- **Resource allocation:** VBHC emphasizes directing resources to interventions that improve outcomes for working-age and early-retirement populations (20-74 years), who have the most to gain from preventive strategies and the longest remaining life expectancy.

- **Regulatory alignment:** The FDA's guidance on digital health technologies for cardiovascular risk stratification similarly focuses on outcomes within 10 years for adults under 75, aligning with this analytical choice.

Thus, restricting the target to natural deaths occurring before age 75 ensures the model predicts what is clinically actionable, not merely biological aging.

In [ ]:
import pandas as pd
import os
import pyreadstat

RAW_PATH = '/content/drive/MyDrive/NHANES_Accelerometry_Project/data/raw'
PROC_PATH = '/content/drive/MyDrive/NHANES_Accelerometry_Project/data/processed'

# Load demographics
demo, meta = pyreadstat.read_xport(os.path.join(RAW_PATH, 'DEMO_C.xpt'))
demo = demo[['SEQN', 'RIDAGEYR']].copy()
demo.columns = ['SEQN', 'age']

print("=== Demographics Loaded ===\n")
print(f"Total rows: {len(demo)}")
print(f"Age range: {demo['age'].min()} - {demo['age'].max()}")
print(f"Age <20: {(demo['age'] < 20).sum()}")
print(f"Age ≥75: {(demo['age'] >= 75).sum()}")

# Merge with eligible (already has target_natural_10yr)
df = eligible.merge(demo, on='SEQN', how='inner')

print(f"\n=== After Merge ===\n")
print(f"Participants: {len(df)}")
print(f"Age range: {df['age'].min()} - {df['age'].max()}")

# Filter: adults 20-74 years
df_filtered = df[(df['age'] >= 20) & (df['age'] < 75)].copy()

print(f"\n=== After Age Filter (20-74 years) ===\n")
print(f"Participants: {len(df_filtered)}")
print(f"Age range: {df_filtered['age'].min()} - {df_filtered['age'].max()}")
print(f"Target events: {df_filtered['target_natural_10yr'].sum()}")
print(f"Event rate: {100 * df_filtered['target_natural_10yr'].mean():.2f}%")

# Preview
print("\n=== First 15 rows ===\n")
print(df_filtered[['SEQN', 'age', 'MORTSTAT', 'PERMTH_EXM', 'UCOD_LEADING', 'target_natural_10yr']].head(15))



In [ ]:
# Save
output = df_filtered[['SEQN', 'target_natural_10yr', 'age']].copy()
output.to_csv(os.path.join(PROC_PATH, 'mortality_target_clean.csv'), index=False)
print(f"\nSaved: {os.path.join(PROC_PATH, 'mortality_target_clean.csv')}")
print(f"Shape: {output.shape}")
print(f"Events: {output['target_natural_10yr'].sum()}")

## 3. Clinical Data Processing

Extract clinical predictors for the 10-year natural mortality model. Sources: DEMO_C (demographics), BMX_C (BMI), BPX_C (blood pressure), L13_C (lipids).

In [ ]:
import pandas as pd
import os
import pyreadstat

PROC_PATH = '/content/drive/MyDrive/NHANES_Accelerometry_Project/data/processed'
RAW_PATH = '/content/drive/MyDrive/NHANES_Accelerometry_Project/data/raw'

# 3.1 Load mortality target (from Step 2.2)
target = pd.read_csv(os.path.join(PROC_PATH, 'mortality_target_clean.csv'))
print(f"Target shape: {target.shape}")
print(f"Events: {target['target_natural_10yr'].sum()}")
print(f"Event rate: {100 * target['target_natural_10yr'].mean():.2f}%")


# 3.2 Demographics (sex, education, income)
demo, _ = pyreadstat.read_xport(os.path.join(RAW_PATH, 'DEMO_C.xpt'))
demo = demo[['SEQN', 'RIAGENDR', 'DMDEDUC2', 'INDFMPIR']].copy()
demo.columns = ['SEQN', 'sex', 'education', 'income']
demo['sex'] = demo['sex'].map({1: 'male', 2: 'female'})
print(f"\nDemographics: {demo.shape}")


# 3.3 BMI
bmx = pd.read_sas(os.path.join(RAW_PATH, 'BMX_C.xpt'), format='xport')
bmx = bmx[['SEQN', 'BMXWT', 'BMXHT']].copy()
bmx.columns = ['SEQN', 'weight_kg', 'height_cm']
bmx['bmi'] = bmx['weight_kg'] / ((bmx['height_cm'] / 100) ** 2)
bmx = bmx.dropna(subset=['bmi'])
print(f"BMI: {bmx.shape}")


# 3.4 Systolic blood pressure
bpx = pd.read_sas(os.path.join(RAW_PATH, 'BPX_C.xpt'), format='xport')
bpx = bpx[['SEQN', 'BPXSY1']].copy()
bpx.columns = ['SEQN', 'sbp']
bpx = bpx.dropna(subset=['sbp'])
print(f"Blood pressure: {bpx.shape}")


# 3.5 Lipids (total cholesterol, HDL)
lipid = pd.read_sas(os.path.join(RAW_PATH, 'L13_C.xpt'), format='xport')
lipid = lipid[['SEQN', 'LBXTC', 'LBXHDD']].copy()
lipid.columns = ['SEQN', 'total_chol', 'hdl']
lipid = lipid.dropna(subset=['total_chol', 'hdl'])
lipid['non_hdl'] = lipid['total_chol'] - lipid['hdl']
print(f"Lipids: {lipid.shape}")


# 3.6 HbA1c (Glycohemoglobin) - Diabetes marker, no fasting required
hba1c = pd.read_sas(os.path.join(RAW_PATH, 'L10_C.xpt'), format='xport')
hba1c = hba1c[['SEQN', 'LBXGH']].copy()
hba1c.columns = ['SEQN', 'hba1c']
print(f"HbA1c: {hba1c.shape}")
print(f"  Range: {hba1c['hba1c'].min():.1f} - {hba1c['hba1c'].max():.1f}%")


# 3.7 Merge all clinical variables
df = target.copy()
df = df.merge(demo, on='SEQN', how='inner')
df = df.merge(bmx, on='SEQN', how='inner')
df = df.merge(bpx, on='SEQN', how='inner')
df = df.merge(lipid, on='SEQN', how='inner')
df = df.merge(hba1c, on='SEQN', how='inner')

print(f"\n=== Final Clinical Dataset ===\n")
print(f"Shape: {df.shape}")
print(f"Participants: {len(df)}")
print(f"Events: {df['target_natural_10yr'].sum()}")
print(f"Event rate: {100 * df['target_natural_10yr'].mean():.2f}%")


# 3.8 Save clinical dataset
df.to_csv(os.path.join(PROC_PATH, 'clinical_data_with_hba1c.csv'), index=False)
print(f"\nSaved: {os.path.join(PROC_PATH, 'clinical_data_with_hba1c.csv')}")

## 4. Accelerometry Data Processing

### 4.1 Overview

The accelerometry data consists of minute-by-minute activity counts recorded over 7 consecutive days. This results in 1,440 variables per participant-day, which is computationally prohibitive and risks severe overfitting if used directly as predictors.

**Strategy:** Aggregate raw minute-level data into clinically meaningful summary metrics that capture volume, intensity, and sedentary behavior.

**Data sources from `rnhanesdata` package:**
- `PAXINTEN_C`: Minute-level activity counts (2003-2004)
- `Flags_C`: Wear/non-wear indicators (1 = valid wear, 0 = non-wear)

### 4.2 Install and Load Required R Packages


In [ ]:
# Install remotes if they are not already installed
if (!require("remotes", quietly = TRUE)) {
  install.packages("remotes")
}

# Install rnhanesdata from GitHub using remotes
if (!require("rnhanesdata", quietly = TRUE)) {
  remotes::install_github("andrew-leroux/rnhanesdata", build_vignettes = FALSE)
  library(rnhanesdata)
}

# Install required packages
library(dplyr)
library(readr)
library(tidyr)

# Check that the data is available
data("PAXINTEN_C")
data("Flags_C")

cat("PAXINTEN_C dimensions:", dim(PAXINTEN_C), "\n")
cat("Flags_C dimensions:", dim(Flags_C), "\n")

In [ ]:
# Check column names (first 10)
colnames(PAXINTEN_C)[1:10]

# Summary of wear time (minutes per day with valid monitor wear)
summary(rowSums(Flags_C[, paste0("MIN", 1:1440)]))

# Preview first 5 days of participant 21005
subset(PAXINTEN_C, SEQN == 21005)[1:5, 1:10]

### 4.3 Apply Non-Wear Mask

In [ ]:
min_cols <- paste0("MIN", 1:1440)
PAXINTEN_C[, min_cols] <- PAXINTEN_C[, min_cols] * Flags_C[, min_cols]

cat("Non-wear mask applied successfully\n")

### 4.4 Calculate Daily Features


In [ ]:
act_mat <- as.matrix(PAXINTEN_C[, min_cols])
flag_mat <- as.matrix(Flags_C[, min_cols])

# Replace NAs with 0 (per vignette recommendation)
act_mat[is.na(act_mat)] <- 0
flag_mat[is.na(flag_mat)] <- 0

# Daily metrics
daily <- PAXINTEN_C[, c("SEQN", "WEEKDAY", "SDDSRVYR")]
daily$TLAC <- rowSums(log(1 + act_mat))           # Total log activity count
daily$wear_time <- rowSums(flag_mat)              # Minutes with valid wear
daily$sedentary <- rowSums(act_mat < 100)         # Minutes with counts <100
daily$mvpa <- rowSums(act_mat >= 2020)            # Minutes with counts >=2020

cat("Daily features calculated\n")
cat("Rows (participant-days):", nrow(daily), "\n")
cat("Columns:", colnames(daily), "\n")

### 4.5 Filter Valid Days (≥600 min wear time)

In [ ]:
daily_valid <- daily[daily$wear_time >= 600, ]

cat("Valid days (>=600 min wear time):", nrow(daily_valid), "\n")
cat("Days excluded:", nrow(daily) - nrow(daily_valid), "\n")
cat("Percentage of days retained:", round(100 * nrow(daily_valid) / nrow(daily), 1), "%\n")


### 4.6 Aggregate by Participant

In [ ]:
participant_features <- daily_valid %>%
  group_by(SEQN) %>%
  summarise(
    TLAC_mean = mean(TLAC, na.rm = TRUE),
    wear_time_mean = mean(wear_time, na.rm = TRUE),
    sedentary_mean = mean(sedentary, na.rm = TRUE),
    mvpa_mean = mean(mvpa, na.rm = TRUE),
    valid_days = n()
  )

cat("Participant-level features:\n")
print(head(participant_features))
cat("\nTotal participants with valid accelerometry data:", nrow(participant_features), "\n")

### 4.7 Save Features to CSV


In [ ]:
output_file <- "/content/drive/MyDrive/NHANES_Accelerometry_Project/data/processed/accelerometry_features.csv"
write.csv(participant_features, output_file, row.names = FALSE)

cat("Saved:", output_file, "\n")
cat("Shape:", dim(participant_features), "\n")

### 4.8 Feature Selection Rationale

Only four accelerometry features are retained for the final model. This conservative choice prevents overfitting given the limited number of events (n = 240) in the clinical dataset.

| Feature | Definition | Clinical Interpretation |
|---------|------------|------------------------|
| `TLAC_mean` | Total log activity count | Overall daily movement volume |
| `sedentary_mean` | Minutes with counts < 100 | Sedentary behavior time |
| `mvpa_mean` | Minutes with counts ≥ 2020 | Moderate-to-vigorous physical activity |
| `valid_days` | Number of days with ≥10h wear | Quality control / compliance |

**Excluded features** (reserved for sensitivity analysis):
- 2-hour window TLACs (12 features) → would exceed events‑per‑predictor ratio
- Fragmentation metrics (SATP, ASTP) → lower clinical interpretability
- Functional PCA components → too complex for this sample size

### 4.9 Summary of Accelerometry Processing

| Step | Output | Value |
|------|--------|-------|
| Raw data | Participant‑days | 50,232 |
| Valid days (≥600 min wear) | Participant‑days | 34,587 (68.9%) |
| Unique participants | Participants | 6,724 |
| Features per participant | 5 (SEQN + 4 metrics + valid_days) | Ready for merge |

The accelerometry features are now saved as `accelerometry_features.csv` and will be merged with the clinical dataset in the next section.

## 5. Merge Clinical and Accelerometry Data

Merge the clinical dataset (with HbA1c) with accelerometry features. Inner join keeps only participants with complete data in both sources.

In [ ]:
import pandas as pd
import numpy as np

PROC_PATH = '/content/drive/MyDrive/NHANES_Accelerometry_Project/data/processed'

# Load clinical data with HbA1c (from Step 3.8)
clinical = pd.read_csv(f"{PROC_PATH}/clinical_data_with_hba1c.csv")
print(f"Clinical data shape: {clinical.shape}")
print(f"Events: {clinical['target_natural_10yr'].sum()}")
print(f"Event rate: {100 * clinical['target_natural_10yr'].mean():.2f}%")

# Load accelerometry features (from Step 4.7)
accel = pd.read_csv(f"{PROC_PATH}/accelerometry_features.csv")
print(f"\nAccelerometry features shape: {accel.shape}")
print(accel.head())

# Merge on SEQN (inner join)
df = clinical.merge(accel, on='SEQN', how='inner')

print(f"\n=== Merged Dataset ===\n")
print(f"Shape: {df.shape}")
print(f"Participants: {len(df)}")
print(f"Events: {df['target_natural_10yr'].sum()}")
print(f"Event rate: {100 * df['target_natural_10yr'].mean():.2f}%")

In [ ]:
df.head(10)

### 5.1 Feature Selection

The following columns are excluded to maintain focus on clinical-biological predictors and avoid redundancy:

| Column | Reason |
|--------|--------|
| `weight_kg`, `height_cm` | Redundant with `bmi` |
| `education`, `income` | Socioeconomic variables (not part of clinical core model) |
| `wear_time_mean` | Quality control, not a biological predictor |

**Retained for modeling:** `age`, `sex`, `bmi`, `sbp`, `total_chol`, `hdl`, `non_hdl`, `hba1c`, `TLAC_mean`, `sedentary_mean`, `mvpa_mean`, `valid_days`.

**Note:** `non_hdl` is retained as a clinically meaningful lipid marker (total cholesterol minus HDL), representing non-HDL cholesterol.

In [ ]:
# Safe drop - only removes columns that exist
columns_to_drop = ['weight_kg', 'height_cm', 'education', 'income', 'wear_time_mean']
columns_existing = [col for col in columns_to_drop if col in df.columns]

df_clean = df.drop(columns=columns_existing)

print(f"Dropped: {columns_existing}")
print(f"Remaining columns: {df_clean.columns.tolist()}")

## 6. Data Quality Validation

Verify data integrity, check for missing values, and validate logical consistency before modeling.

In [ ]:
print("=== Missing Values Check ===\n")
missing_all = df_clean.isnull().sum()
missing_pct = 100 * missing_all / len(df_clean)
missing_report = pd.DataFrame({
    'missing_count': missing_all,
    'missing_pct': missing_pct
}).sort_values('missing_count', ascending=False)
print(missing_report[missing_report['missing_count'] > 0])

In [ ]:
# Drop rows with missing hba1c
df_clean = df_clean.dropna(subset=['hba1c'])

print(f"Shape after dropping 4 missing hba1c: {df_clean.shape}")
print(f"Events: {df_clean['target_natural_10yr'].sum()}")
print(f"Event rate: {100 * df_clean['target_natural_10yr'].mean():.2f}%")

In [ ]:
print("=== Range Validation ===\n")

# Age (should be 20-74)
print(f"Age range: {df_clean['age'].min():.0f} - {df_clean['age'].max():.0f}")
assert df_clean['age'].min() >= 20 and df_clean['age'].max() <= 74

# BMI
print(f"BMI range: {df_clean['bmi'].min():.1f} - {df_clean['bmi'].max():.1f}")
assert df_clean['bmi'].min() >= 10 and df_clean['bmi'].max() <= 70

# SBP
print(f"SBP range: {df_clean['sbp'].min():.0f} - {df_clean['sbp'].max():.0f} mmHg")
assert df_clean['sbp'].min() >= 50 and df_clean['sbp'].max() <= 250

# HbA1c
print(f"HbA1c range: {df_clean['hba1c'].min():.1f} - {df_clean['hba1c'].max():.1f}%")

# Total cholesterol
print(f"Total cholesterol range: {df_clean['total_chol'].min():.0f} - {df_clean['total_chol'].max():.0f} mg/dL")

# HDL
print(f"HDL range: {df_clean['hdl'].min():.0f} - {df_clean['hdl'].max():.0f} mg/dL")

### 6.1 Range Validation Results

All continuous variables fall within clinically plausible ranges:

| Variable | Range | Clinical Plausibility |
|----------|-------|----------------------|
| Age | 20 - 74 years | ✅ Expected (filtered) |
| BMI | 14.7 - 57.7 | ✅ Acceptable (underweight to severe obesity) |
| SBP | 80 - 218 mmHg | ✅ Normal to severe hypertension |
| HbA1c | 4.0 - 18.0% | ✅ Normal to poorly controlled diabetes |
| Total cholesterol | 83 - 650 mg/dL | ✅ Normal to extreme hyperlipidemia |
| HDL | 22 - 154 mg/dL | ✅ Low to high (low is risk factor) |

**No out-of-range values detected.** All checks passed.

## 7. Final Dataset for Modeling

Save the cleaned, validated dataset for use in the modeling notebook (`02_model_training_evaluation.ipynb`).

In [ ]:
print("=== Final Dataset Summary ===\n")
print(f"Shape: {df_clean.shape}")
print(f"Participants: {len(df_clean)}")
print(f"Events (10-year natural mortality): {df_clean['target_natural_10yr'].sum()}")
print(f"Event rate: {100 * df_clean['target_natural_10yr'].mean():.2f}%")
print(f"\nFeatures for modeling:")
print(f"  - Demographic: age, sex")
print(f"  - Clinical: bmi, sbp, total_chol, hdl, non_hdl, hba1c")
print(f"  - Accelerometry: TLAC_mean, sedentary_mean, mvpa_mean, valid_days")
print(f"  - Total predictors: {len(df_clean.columns) - 2} (excluding SEQN and target)")
print(f"  - Events per predictor: {df_clean['target_natural_10yr'].sum() / (len(df_clean.columns) - 2):.1f}")

In [ ]:
# Save final dataset (already cleaned)
output_path = f"{PROC_PATH}/nhanes_2003_2004_clean.csv"
df_clean.to_csv(output_path, index=False)

print(f" Saved: {output_path}")